# Ringkasan Otomatis Materi PKn dengan IndoBERT (Extractive Summarization)

Notebook ini melatih **IndoBERT** (`indobenchmark/indobert-base-p1`) untuk meringkas materi PKn secara **ekstraktif**, yaitu memilih kalimat-kalimat paling penting dari teks asli untuk dijadikan ringkasan.

Pipeline notebook ini sama dengan notebook DistilBERT (`PKN with distillBERT.Ipynb`), sehingga hasil kedua model dapat dibandingkan secara adil.

**Mengapa ekstraktif?**
Dataset memiliki kolom `teks_materi` (teks asli) dan `rangkuman_materi` (ringkasan referensi). Hampir seluruh kalimat pada `rangkuman_materi` ditemukan persis (verbatim) di `teks_materi`, jadi dataset ini cocok untuk pendekatan ekstraktif.

**Alur kerja:**
1. Pecah `teks_materi` menjadi kalimat-kalimat.
2. Beri label `1` jika kalimat tersebut juga ada di `rangkuman_materi`, `0` jika tidak.
3. Fine-tune IndoBERT sebagai **binary sentence classifier** (apakah kalimat layak masuk ringkasan).
4. Saat inferensi, hitung skor tiap kalimat pada dokumen baru, lalu ambil kalimat berskor tertinggi (urut sesuai posisi asli) sebagai ringkasan.

Model hasil training disimpan ke folder lokal `./model_indobert_pkn_summarizer` agar dapat dibandingkan dengan model DistilBERT atau dipakai sebagai backend website peringkas e-book PKn.

> Catatan: IndoBERT base (±110M parameter) lebih besar dari DistilBERT (±66M), jadi `per_device_train_batch_size` diturunkan dan `gradient_accumulation_steps` digunakan agar tetap muat di VRAM yang terbatas (±4GB).
>
> Jika muncul error terkait `Trainer`/`accelerate`, coba jalankan `%pip install -U transformers accelerate` lalu restart kernel.

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────────────
!pip install transformers datasets torch pandas scikit-learn rouge-score sentencepiece accelerate -q

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score,
    classification_report,
    confusion_matrix,
)

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
from rouge_score import rouge_scorer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device      : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
    print(f'CUDA version: {torch.version.cuda}')
    print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    torch.backends.cudnn.benchmark = True
else:
    print('CUDA ga kedeteksi — jalan di CPU')

Device      : cuda
GPU         : Tesla T4
CUDA version: 12.8
VRAM        : 15.6 GB


## 1. Load Dataset

File `dataset_pkn_fix.csv` menggunakan encoding `cp1252` (bukan UTF-8), jadi perlu ditentukan saat membaca.

In [ ]:
DATA_PATH = "dataset_pkn_fix.csv"

df = pd.read_csv(DATA_PATH, encoding="cp1252")
print("Jumlah dokumen (sub-bab):", len(df))
df[["id_data", "jenjang", "kelas", "judul_bab", "judul_sub_bab"]].head()

Jumlah dokumen (sub-bab): 307


,id_data,jenjang,kelas,judul_bab,judul_sub_bab
0,PKN-VII-2021-BI-P,SMP,VII,Sejarah Kelahiran Pancasila,Pendahuluan
1,PKN-VII-2021-BI-A,SMP,VII,Sejarah Kelahiran Pancasila,Latar Sejarah Kelahiran Pancasila
2,PKN-VII-2021-BI-B,SMP,VII,Sejarah Kelahiran Pancasila,Kelahiran Pancasila
3,PKN-VII-2021-BI-C,SMP,VII,Sejarah Kelahiran Pancasila,Perumusan Pancasila
4,PKN-VII-2021-BI-D,SMP,VII,Sejarah Kelahiran Pancasila,Penetapan Pancasila


## 2. Praproses: Pemecahan Kalimat & Pelabelan

Setiap dokumen (`teks_materi`) dipecah menjadi kalimat-kalimat. Sebuah kalimat diberi label:
- `1` jika kalimat tersebut juga muncul (verbatim) di `rangkuman_materi`
- `0` jika tidak

Hasilnya adalah dataset baru pada level **kalimat**, yang digunakan untuk fine-tuning IndoBERT sebagai sentence classifier.

In [ ]:
SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def split_sentences(text):
    text = re.sub(r"\s+", " ", str(text)).strip()
    sentences = [s.strip() for s in SENT_SPLIT_RE.split(text) if s.strip()]
    return [s for s in sentences if len(s) > 3]


# contoh
split_sentences(df.iloc[0]["teks_materi"])[:3]

['Kalian tentu tahu burung Garuda.',
 'Burung yang gambarnya dijadikan lambang negara Indonesia, dengan simbol Pancasila di dadanya.',
 'Garuda adalah nama burung yang ada dalam cerita wayang.']

In [ ]:
records = []
for _, row in df.iterrows():
    src_sentences = split_sentences(row["teks_materi"])
    sum_sentences = set(split_sentences(row["rangkuman_materi"]))

    for pos, sent in enumerate(src_sentences):
        records.append({
            "id_data": row["id_data"],
            "sentence": sent,
            "position": pos,
            "doc_len": len(src_sentences),
            "label": int(sent in sum_sentences),
        })

sent_df = pd.DataFrame(records)
print("Total kalimat:", len(sent_df))
print(sent_df["label"].value_counts())
print(sent_df["label"].value_counts(normalize=True))
sent_df.head()

Total kalimat: 25286
label
0    22591
1     2695
Name: count, dtype: int64
label
0    0.893419
1    0.106581
Name: proportion, dtype: float64


,id_data,sentence,position,doc_len,label
0,PKN-VII-2021-BI-P,Kalian tentu tahu burung Garuda.,0,28,1
1,PKN-VII-2021-BI-P,Burung yang gambarnya dijadikan lambang negara...,1,28,1
2,PKN-VII-2021-BI-P,Garuda adalah nama burung yang ada dalam cerit...,2,28,1
3,PKN-VII-2021-BI-P,Burung itu merupakan anak dewa yang menjadi tu...,3,28,0
4,PKN-VII-2021-BI-P,"Di alam nyata, burung Garuda dalam cerita ters...",4,28,1


## 3. Split Train / Validation / Test

Pembagian dilakukan **per dokumen** (bukan per kalimat) agar kalimat dari dokumen yang sama tidak bocor antar split.

In [ ]:
doc_ids = df["id_data"].unique()

train_ids, temp_ids = train_test_split(doc_ids, test_size=0.2, random_state=SEED)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=SEED)

train_df = sent_df[sent_df["id_data"].isin(train_ids)].reset_index(drop=True)
val_df = sent_df[sent_df["id_data"].isin(val_ids)].reset_index(drop=True)
test_df = sent_df[sent_df["id_data"].isin(test_ids)].reset_index(drop=True)

print(f"Train: {len(train_df)} kalimat dari {len(train_ids)} dokumen")
print(f"Val  : {len(val_df)} kalimat dari {len(val_ids)} dokumen")
print(f"Test : {len(test_df)} kalimat dari {len(test_ids)} dokumen")

Train: 19630 kalimat dari 245 dokumen
Val  : 2871 kalimat dari 31 dokumen
Test : 2785 kalimat dari 31 dokumen


## 4. Tokenisasi

Model dasar yang digunakan: **`indobenchmark/indobert-base-p1`**, yaitu BERT base yang sudah dipre-train pada korpus Bahasa Indonesia (IndoNLU).

In [ ]:
MODEL_NAME = "flax-community/indonesian-roberta-base"
MAX_LEN = 96

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def to_hf_dataset(d):
    return Dataset.from_pandas(
        d[["sentence", "label"]].rename(columns={"label": "labels"}),
        preserve_index=False,
    )


train_ds = to_hf_dataset(train_df)
val_ds = to_hf_dataset(val_df)
test_ds = to_hf_dataset(test_df)


def tokenize_fn(batch):
    return tokenizer(batch["sentence"], truncation=True, max_length=MAX_LEN)


train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/19630 [00:00<?, ? examples/s]

Map:   0%|          | 0/2871 [00:00<?, ? examples/s]

Map:   0%|          | 0/2785 [00:00<?, ? examples/s]

## 5. Model & Penanganan Ketidakseimbangan Kelas

Hanya sekitar 10-15% kalimat yang masuk ringkasan, sehingga kelas `1` jauh lebih jarang dibanding kelas `0`. Untuk mengatasinya, digunakan **class weights** pada `CrossEntropyLoss`.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True,
    output_hidden_states=False,
    output_attentions=False,
).to(DEVICE)

class_counts = train_df["label"].value_counts().sort_index()
class_weights = torch.tensor(
    [len(train_df) / (2.0 * c) for c in class_counts],
    dtype=torch.float,
).to(DEVICE)
print("Jumlah per kelas:", class_counts.to_dict())
print("Class weights   :", class_weights.tolist())


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: flax-community/indonesian-roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Jumlah per kelas: {0: 17636, 1: 1994}
Class weights   : [0.556532084941864, 4.922266960144043]


## 6. Training

IndoBERT base lebih besar dari DistilBERT, sehingga `per_device_train_batch_size` diturunkan menjadi `8` dan `gradient_accumulation_steps=2` digunakan agar batch efektif tetap `16` tanpa kehabisan VRAM. `EarlyStoppingCallback` digunakan agar training berhenti otomatis saat F1 validasi tidak membaik lagi.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


training_args = TrainingArguments(
    output_dir="./checkpoints/roberta-pkn-extractive",

    num_train_epochs=10,
    learning_rate=2e-5,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    warmup_ratio=0.1,
    lr_scheduler_type="cosine",

    logging_steps=50,
    save_total_limit=2,

    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2)
    ],
)

trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.034513,0.604619,0.827238,0.449219,0.518018,0.481172
2,0.843525,1.067497,0.842564,0.490909,0.486486,0.488688
3,0.842649,0.917717,0.796238,0.400000,0.635135,0.490862
4,0.884804,1.963501,0.842564,0.490991,0.490991,0.490991
5,0.394742,2.274123,0.847092,0.506173,0.461712,0.482921
6,0.133330,2.628023,0.845350,0.500000,0.468468,0.483721


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7362, training_loss=0.7123292285895354, metrics={'train_runtime': 1127.6725, 'train_samples_per_second': 139.26, 'train_steps_per_second': 8.705, 'total_flos': 2994796837648920.0, 'train_loss': 0.7123292285895354, 'epoch': 6.0})

## 7. Evaluasi Sentence-Level (Validation & Test Set)

Mengukur seberapa baik model menebak kalimat mana yang seharusnya masuk ringkasan.

In [ ]:
results = trainer.evaluate()

print("\n=== HASIL VALIDATION ===")
print(results)

preds_output = trainer.predict(val_ds)
y_pred = np.argmax(preds_output.predictions, axis=1)
y_true = preds_output.label_ids

print("\n=== CLASSIFICATION REPORT (VALIDATION) ===")
print(classification_report(y_true, y_pred, digits=4))

print("\n=== CONFUSION MATRIX (VALIDATION) ===")
print(confusion_matrix(y_true, y_pred))

print("\n=== BEST CHECKPOINT ===")
print(trainer.state.best_model_checkpoint)
print("Best F1:", trainer.state.best_metric)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.133330,1.963501,6,0.842564,0.490991,0.490991,0.490991



=== HASIL VALIDATION ===
{'eval_loss': 1.9635013341903687, 'eval_accuracy': 0.8425635667014978, 'eval_precision': 0.49099099099099097, 'eval_recall': 0.49099099099099097, 'eval_f1': 0.49099099099099097}



=== CLASSIFICATION REPORT (VALIDATION) ===
              precision    recall  f1-score   support

           0     0.9069    0.9069    0.9069      2427
           1     0.4910    0.4910    0.4910       444

    accuracy                         0.8426      2871
   macro avg     0.6989    0.6989    0.6989      2871
weighted avg     0.8426    0.8426    0.8426      2871


=== CONFUSION MATRIX (VALIDATION) ===
[[2201  226]
 [ 226  218]]

=== BEST CHECKPOINT ===
./checkpoints/roberta-pkn-extractive/checkpoint-4908
Best F1: 0.49099099099099097


In [ ]:
test_metrics = trainer.evaluate(test_ds)
test_metrics

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.133330,1.753392,6,0.883303,0.326531,0.249027,0.282561


{'eval_loss': 1.7533915042877197,
 'eval_accuracy': 0.8833034111310593,
 'eval_precision': 0.32653061224489793,
 'eval_recall': 0.2490272373540856,
 'eval_f1': 0.282560706401766}

## 8. Evaluasi Ringkasan End-to-End (ROUGE)

Untuk setiap dokumen di test set, bentuk ringkasan dengan memilih kalimat berskor tertinggi (sesuai rasio kompresi), lalu bandingkan dengan `rangkuman_materi` menggunakan ROUGE.

In [ ]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)


def predict_sentence_scores(sentences, batch_size=32):
    model.eval()
    scores = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i + batch_size]
        enc = tokenizer(
            batch, truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt"
        ).to(DEVICE)
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1)[:, 1]
        scores.extend(probs.cpu().tolist())
    return scores


def extractive_summarize(text, ratio=0.15, min_sentences=3):
    sentences = split_sentences(text)
    if len(sentences) <= min_sentences:
        return " ".join(sentences)

    scores = predict_sentence_scores(sentences)
    n_select = max(min_sentences, round(len(sentences) * ratio))
    top_idx = sorted(range(len(sentences)), key=lambda i: scores[i], reverse=True)[:n_select]
    top_idx = sorted(top_idx)
    return " ".join(sentences[i] for i in top_idx)


rouge1_scores, rouge2_scores, rougeL_scores = [], [], []
test_docs = df[df["id_data"].isin(test_ids)]

for _, row in test_docs.iterrows():
    pred_summary = extractive_summarize(row["teks_materi"], ratio=0.15)
    ref_summary = row["rangkuman_materi"]
    rouge = scorer.score(ref_summary, pred_summary)
    rouge1_scores.append(rouge["rouge1"].fmeasure)
    rouge2_scores.append(rouge["rouge2"].fmeasure)
    rougeL_scores.append(rouge["rougeL"].fmeasure)

print("ROUGE-1 (F1):", np.mean(rouge1_scores))
print("ROUGE-2 (F1):", np.mean(rouge2_scores))
print("ROUGE-L (F1):", np.mean(rougeL_scores))

ROUGE-1 (F1): 0.5560237796325981
ROUGE-2 (F1): 0.4632185812905171
ROUGE-L (F1): 0.48711279136276764


## 9. Simpan Model untuk Deployment

Model dan tokenizer disimpan ke folder lokal agar dapat dimuat kembali oleh backend website (misalnya Flask/FastAPI).

In [ ]:
SAVE_DIR = "./model_roberta_pkn_summarizer"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Model & tokenizer tersimpan di:", SAVE_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model & tokenizer tersimpan di: ./model_roberta_pkn_summarizer


## 10. Demo Inferensi

Contoh meringkas salah satu dokumen, dibandingkan dengan ringkasan asli pada dataset.

In [ ]:
sample = df.iloc[0]

print("=== JUDUL ===")
print(sample["judul_bab"], "-", sample["judul_sub_bab"])

print("\n=== RINGKASAN MODEL (RoBERTa) ===")
print(extractive_summarize(sample["teks_materi"], ratio=0.15))

print("\n=== RINGKASAN ASLI (DATASET) ===")
print(sample["rangkuman_materi"])

=== JUDUL ===
Sejarah Kelahiran Pancasila - Pendahuluan

=== RINGKASAN MODEL (RoBERTa) ===
Garuda adalah nama burung yang ada dalam cerita wayang. Di alam nyata, burung Garuda dalam cerita tersebut adalah burung rajawali atau burung elang besar. Maka rajawali atau elang memang layak dijadikan lambang negara Indonesia. Dalam pembelajaran pertama PPKn kali ini kalian diajak lebih dahulu mengenal burung Garuda, burung yang dijadikan lambang negara Republik Indonesia.

=== RINGKASAN ASLI (DATASET) ===
Kalian tentu tahu burung Garuda. Burung yang gambarnya dijadikan lambang negara Indonesia, dengan simbol Pancasila di dadanya. Garuda adalah nama burung yang ada dalam cerita wayang. Di alam nyata, burung Garuda dalam cerita tersebut adalah burung rajawali atau burung elang besar. Maka rajawali atau elang memang layak dijadikan lambang negara Indonesia. Dalam pembelajaran pertama PPKn kali ini kalian diajak lebih dahulu mengenal burung Garuda, burung yang dijadikan lambang negara Republik Ind

## 11. Memuat Ulang Model di Backend Website

Kode berikut dipakai di backend (Flask/FastAPI) untuk memuat model yang sudah disimpan, lalu memanggil `extractive_summarize` untuk meringkas teks e-book PKn yang diunggah pengguna.

```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

SAVE_DIR = "./model_indobert_pkn_summarizer"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to(DEVICE)
model.eval()

# gunakan fungsi extractive_summarize(text, ratio=0.15) dari notebook ini
```

> Bandingkan `eval_f1` (Bagian 7) dan skor ROUGE (Bagian 8) dengan hasil pada `PKN with distillBERT.Ipynb` untuk menentukan model mana yang lebih cocok dipakai sebagai backend.